In [1]:
import os

import pandas as pd
import timm
import torch
from PIL import Image
from sklearn.model_selection import train_test_split
from torch import nn, optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from tqdm import tqdm


In [2]:
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")

Using device: cuda


In [3]:
# Custom Dataset for Test Data
class GTSRBTestDataset(Dataset):
    def __init__(self, annotations, root_dir, transform=None):
        """
        Args:
            csv_file (str): Path to the CSV file with image filenames and labels.
            root_dir (str): Directory with all the images.
            transform (callable, optional): Optional transform to be applied on an image.
        """
        self.annotations = annotations
        self.root_dir = root_dir
        self.transform = transform

    def __len__(self):
        return len(self.annotations)

    def __getitem__(self, idx):
        img_name = os.path.join(self.root_dir, self.annotations["Path"][idx])
        image = Image.open(img_name).convert("RGB")
        label = self.annotations["ClassId"][idx]

        if self.transform:
            image = self.transform(image)

        return image, label

In [4]:
data_dir = "/kaggle/input/gtsrb-german-traffic-sign/"  # Path to the GTSRB dataset

train_val_annotations = pd.read_csv(os.path.join(data_dir, "Train.csv"))  # CSV file with training image names and labels
test_anotations = pd.read_csv(os.path.join(data_dir, "Test.csv"))  # CSV file with test image names and labels

# Train test split
train_annotations, val_annotations_split = train_test_split(
    train_val_annotations, test_size=0.2, random_state=42
)

train_annotations = train_annotations.reset_index(drop=True)
val_annotations_split = val_annotations_split.reset_index(drop=True)
test_annotations = test_anotations.reset_index(drop=True)

In [6]:
batch_size = 128
num_classes = 43

# Define data transformations with augmentations for training
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),  # Randomly flip images
    transforms.RandomRotation(degrees=30),  # Randomly rotate images
    transforms.RandomResizedCrop(size=(224, 224), scale=(0.8, 1.0)),  # Random crop and resize
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# Define transformations for testing (no augmentations)
test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# Load training dataset using custom loader
train_dataset = GTSRBTestDataset(annotations=train_annotations, root_dir=data_dir, transform=train_transform)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

# Load validation dataset using custom loader
val_dataset = GTSRBTestDataset(annotations=val_annotations_split, root_dir=data_dir, transform=test_transform)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

# Load test dataset using custom loader
test_dataset = GTSRBTestDataset(annotations=test_anotations, root_dir=data_dir, transform=test_transform)
test__loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [8]:
# Define the training function
def train_single_epoch(model, train_loader, criterion, optimizer, device, epoch):
    model.train()  # Set the model to training mode
    running_loss = 0.0
    correct = 0
    total = 0

    # Iterate over the training data
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}")
    for inputs, labels in progress_bar:
        inputs, labels = inputs.to(device), labels.to(device)

        # Ensure labels are 1D (class indices)
        if labels.ndim > 1:
            labels = labels.squeeze()  # Remove extra dimensions if necessary

        # Zero the parameter gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(inputs)

        # Ensure outputs are in the correct shape for CrossEntropyLoss
        # (batch_size, num_classes)
        if outputs.ndim > 2:
            outputs = outputs.view(outputs.size(0), -1)

        loss = criterion(outputs, labels)

        # Backward pass and optimization
        loss.backward()
        optimizer.step()

        # Track loss and accuracy
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

        # Update tqdm bar with current loss
        progress_bar.set_postfix(loss=loss.item())


def validate_model(model, val_loader, criterion, device):
    model.eval()  # Set the model to evaluation mode
    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():  # Disable gradient computation for validation
        progress_bar = tqdm(val_loader, desc="Validation")
        for inputs, labels in progress_bar:
            inputs, labels = inputs.to(device), labels.to(device)

            # Ensure labels are 1D (class indices)
            if labels.ndim > 1:
                labels = labels.squeeze()  # Remove extra dimensions if necessary

            # Forward pass
            outputs = model(inputs)

            # Ensure outputs are in the correct shape for CrossEntropyLoss
            if outputs.ndim > 2:
                outputs = outputs.view(outputs.size(0), -1)

            loss = criterion(outputs, labels)

            # Track loss and accuracy
            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

            # Update tqdm bar with current loss
            progress_bar.set_postfix(loss=loss.item())

    # Calculate validation loss and accuracy
    val_loss = running_loss / len(val_loader)
    val_acc = 100.0 * correct / total
    return val_loss, val_acc

def train(model, train_loader, criterion, optimizer, device, num_epochs):
    for epoch in range(num_epochs):
        train_single_epoch(model, train_loader, criterion, optimizer, device, epoch)
        val_loss, val_acc = validate_model(model, val_loader, criterion, device)
        print(f"Epoch {epoch+1}/{num_epochs}, Validation Loss: {val_loss:.4f}, Validation Accuracy: {val_acc:.2f}%")

In [10]:
# Training parameters
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
num_epochs = 10

# Load the Swin Transformer model
model = timm.create_model("swin_tiny_patch4_window7_224", pretrained=True, num_classes=num_classes)
model = model.to(device)  # Move the model to the appropriate device (GPU/CPU)

# Define the loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=2e-4, weight_decay=1e-4)

# Call the training function
train(model, train_loader, criterion, optimizer, device, num_epochs)

# Save the trained model
torch.save(model.state_dict(), "swin_tiny_gtsrb.pth")
print("Model saved as swin_tiny_gtsrb.pth")

Validation: 100%|██████████| 62/62 [01:38<00:00,  1.59s/it, loss=0.0114]


Epoch 1/10, Validation Loss: 0.0526, Validation Accuracy: 98.23%


Validation: 100%|██████████| 62/62 [00:44<00:00,  1.41it/s, loss=0.0313] 


Epoch 2/10, Validation Loss: 0.0411, Validation Accuracy: 98.71%


Validation: 100%|██████████| 62/62 [00:42<00:00,  1.44it/s, loss=0.0103]  


...

Epoch 8/10, Validation Loss: 0.0216, Validation Accuracy: 99.29%


Validation: 100%|██████████| 62/62 [00:44<00:00,  1.38it/s, loss=0.000411]


Epoch 9/10, Validation Loss: 0.0085, Validation Accuracy: 99.76%


Validation: 100%|██████████| 62/62 [00:42<00:00,  1.45it/s, loss=2.91e-5] 

Epoch 10/10, Validation Loss: 0.0060, Validation Accuracy: 99.81%
Model saved as swin_tiny_gtsrb.pth
